# Milestone 3 - Feature Selection Experiments

## Objective

Evaluate whether adding `city`, `updates`, or both improves model performance compared with the original Milestone 3 feature set.

To make the comparison fair, all experiments use the same:
- Dataset
- Train/test split
- Logistic Regression model
- Preprocessing approach
- Evaluation metrics

The only difference between experiments is the feature set used.

### Note on `updates`

The `updates` feature is included as an exploratory feature based on Challenge Advisor guidance. Because the historical dataset may include updates posted during the campaign, this feature may contain post-launch information.

For a true pre-launch recommendation system, `updates` would need to represent planned updates or milestones known before launch.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [2]:
# Load the dataset and create target 
df = pd.read_csv("../data/kickstarter_us_filtered.csv")

# Binary target
df["success"] = (df["status"] == "successful").astype(int)

print("Dataset shape:", df.shape)

Dataset shape: (38504, 18)


In [3]:
# Extract city and state from location column
df[["city", "state"]] = df["location"].str.rsplit(
    ",",
    n=1,
    expand=True
)

df["city"] = df["city"].str.strip()
df["state"] = df["state"].str.strip()

# Inspect them
print("Unique cities:", df["city"].nunique())
print("Unique states:", df["state"].nunique())

print("\nMissing city values:", df["city"].isnull().sum())
print("Missing state values:", df["state"].isnull().sum())
print("Missing updates values:", df["updates"].isnull().sum())

Unique cities: 3142
Unique states: 51

Missing city values: 0
Missing state values: 0
Missing updates values: 0


In [4]:
# Define the 4 feature sets for experiments
feature_sets = {
    "Baseline": {
        "numerical": [
            "goal",
            "duration",
            "levels"
        ],
        "categorical": [
            "category",
            "subcategory",
            "state"
        ]
    },

    "City Only": {
        "numerical": [
            "goal",
            "duration",
            "levels"
        ],
        "categorical": [
            "category",
            "subcategory",
            "state",
            "city"
        ]
    },

    "Updates Only": {
        "numerical": [
            "goal",
            "duration",
            "levels",
            "updates"
        ],
        "categorical": [
            "category",
            "subcategory",
            "state"
        ]
    },

    "City + Updates": {
        "numerical": [
            "goal",
            "duration",
            "levels",
            "updates"
        ],
        "categorical": [
            "category",
            "subcategory",
            "state",
            "city"
        ]
    }
}

In [5]:
# Create 1 shared train-test split
y = df["success"].copy()

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(train_idx))
print("Testing rows:", len(test_idx))

Training rows: 30803
Testing rows: 7701


In [6]:
# Create 1 function that trains and evaluates each experiment
def run_logistic_experiment(numerical_features, categorical_features):

    selected_features = numerical_features + categorical_features

    # Use the exact same train/test rows for every experiment
    X_train = df.loc[train_idx, selected_features].copy()
    X_test = df.loc[test_idx, selected_features].copy()

    y_train = df.loc[train_idx, "success"].copy()
    y_test = df.loc[test_idx, "success"].copy()

    # Preprocessing
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numerical_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"),
             categorical_features)
        ]
    )

    X_train_processed = preprocessor.fit_transform(X_train)
    X_test_processed = preprocessor.transform(X_test)

    # Logistic Regression
    model = LogisticRegression(max_iter=1000)

    model.fit(X_train_processed, y_train)

    # Predictions
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]

    # Return results
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Processed Features": X_train_processed.shape[1]
    }

In [7]:
# Run all 4 experiments and store results
results = []

for experiment_name, features in feature_sets.items():

    metrics = run_logistic_experiment(
        numerical_features=features["numerical"],
        categorical_features=features["categorical"]
    )

    metrics["Experiment"] = experiment_name
    results.append(metrics)

In [8]:
# Turn the results into a table
results_df = pd.DataFrame(results)

results_df = results_df[
    [
        "Experiment",
        "Processed Features",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ]
]

results_df.round(4)

,Experiment,Processed Features,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Baseline,119,0.6860,0.6910,0.7713,0.7289,0.7463
1,City Only,2936,0.6907,0.6959,0.7725,0.7322,0.7534
2,Updates Only,120,0.8025,0.8203,0.8185,0.8194,0.8804
3,City + Updates,2937,0.8041,0.8221,0.8192,0.8207,0.8801


In [9]:
# Calculate improvements over the baseline
baseline_accuracy = results_df.loc[
    results_df["Experiment"] == "Baseline",
    "Accuracy"
].iloc[0]

baseline_roc_auc = results_df.loc[
    results_df["Experiment"] == "Baseline",
    "ROC-AUC"
].iloc[0]

# Create a shorter improvement table
improvement_df = results_df[
    ["Experiment", "Accuracy", "ROC-AUC"]
].copy()

improvement_df["Accuracy Change (Percentage Points)"] = (
    (improvement_df["Accuracy"] - baseline_accuracy) * 100
)

improvement_df["ROC-AUC Change"] = (
    improvement_df["ROC-AUC"] - baseline_roc_auc
)

# Keep only the columns we want to display
improvement_df = improvement_df[
    [
        "Experiment",
        "Accuracy Change (Percentage Points)",
        "ROC-AUC Change"
    ]
]

display(improvement_df.round(4))

,Experiment,Accuracy Change (Percentage Points),ROC-AUC Change
0,Baseline,0.0000,0.0000
1,City Only,0.4675,0.0071
2,Updates Only,11.6478,0.1341
3,City + Updates,11.8037,0.1338


## Findings

- Adding `city` improved accuracy from **68.60%** to **69.07%**, an increase of about **0.47 percentage points.** However, the number of processed features increased from **119 to 2,936** because of the high cardinality of city. Given the relatively small performance improvement compared with the added model complexity and computational cost, I do not think adding city alone provides enough benefit to justify including it.

- Adding `updates` produced a much larger improvement. Accuracy increased from **68.60% to 80.25%**, a gain of about **11.65 percentage points**, while ROC-AUC increased from **0.7463 to 0.8804**. Based on both the performance improvement and the Challenge Advisor's guidance that campaign updates are an important signal, `updates` appears to provide substantial predictive value. However, the limitation around whether historical updates contain post-launch information should still be documented.

- Combining `city` and `updates` produced the highest accuracy at **80.41%**, but this was only about **0.16 percentage points** higher than using `updates` alone. It also increased the number of processed features from **120 to 2,937**, while ROC-AUC slightly decreased from **0.8804 to 0.8801**. Because the additional performance gain from city is very small compared with the increase in model complexity, I would currently prefer the `updates` only feature set over `city + updates`.